In [ ]:
# ── Model B: Surprise-Only Propagation ───────────────────────────────
# Hypothesis: only prediction error propagates between layers.
# Raw activations never leave their layer.
# Computation scales with surprise, not input size.

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
from PIL import Image
import os
from sklearn.metrics import classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Core Component: Predictive Layer ─────────────────────────────────
# Each layer does three things:
# 1. Encodes incoming signal (bottom-up)
# 2. Predicts what incoming signal should look like (prediction)
# 3. Computes error = actual - predicted (what gets passed up)

class PredictiveLayer(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        # bottom-up encoder — standard conv
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
        # predictor — tries to reconstruct incoming signal
        # takes encoded representation and projects back to input space
        self.predictor = nn.Sequential(
            nn.Conv2d(out_channels, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )
        
        # error encoder — encodes the surprise signal
        # this is what actually propagates upward
        self.error_encoder = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
        # scale factor — learned gating of how much surprise to pass up
        # if surprise is small, gate it down — dont waste compute above
        self.surprise_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(out_channels, out_channels),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        # encode input
        encoded = self.encoder(x)
        
        # predict what input should have looked like
        prediction = self.predictor(encoded)
        
        # match spatial size if stride changed it
        if prediction.shape != x.shape:
            prediction = F.interpolate(prediction, size=x.shape[2:])
        
        # compute surprise — what was unexpected
        error = x - prediction
        
        # encode the error signal
        error_encoded = self.error_encoder(error)
        
        # gate by surprise magnitude
        # high surprise = gate open, low surprise = gate closed
        gate = self.surprise_gate(error_encoded)
        gate = gate.unsqueeze(-1).unsqueeze(-1)
        gated_error = error_encoded * gate
        
        # return gated error (propagates up) and prediction (for loss)
        return gated_error, prediction, error

# test single layer
layer = PredictiveLayer(3, 32).to(device)
test_input = torch.randn(2, 3, 224, 224).to(device)
out, pred, err = layer(test_input)
print(f"Input shape:            {test_input.shape}")
print(f"Propagated error shape: {out.shape}")
print(f"Prediction shape:       {pred.shape}")
print(f"Raw error shape:        {err.shape}")
print(f"Mean surprise magnitude: {err.abs().mean().item():.4f}")